# Baseline Model - ResNet34 U-Net

Training + evaluation notebook for the baseline segmentation model. This is intentionally a work in progress: use the smoke checks while prototyping, then uncomment full training when the data path and cloud runtime are ready.

## Colab Setup

Run these when using Colab: mounts Drive, moves to repo, install dependencies from `requirements.txt`, and loads W&B auth from Colab Secrets (see README if not setup).

In [ ]:
# # Colab only: mount Drive and move into the repo.
# from google.colab import drive

# drive.mount("/content/drive")
# %cd /content/drive/MyDrive/Deep Learning Project/DLE-Flair-Segmentation

In [ ]:
# # Colab only: install dependencies into the runtime.
# !pip install -r requirements.txt

In [ ]:
# # Colab only: load W&B API key from Colab Secrets.
# # In Colab, add a secret named exactly WANDB_API_KEY before running this cell.
# import os
# from google.colab import userdata

# wandb_api_key = userdata.get("WANDB_API_KEY")
# if wandb_api_key:
#     os.environ["WANDB_API_KEY"] = wandb_api_key
#     print("WANDB_API_KEY loaded from Colab Secrets.")
# else:
#     print("WANDB_API_KEY was not found in Colab Secrets; W&B training will fail if enabled.")

## Local Setup

Start here when local. In Colab, run this section **after** the Colab setup above.

In [ ]:
from pathlib import Path
import os
import sys

NOTEBOOK_DIR = Path.cwd()
REPO_ROOT = NOTEBOOK_DIR if (NOTEBOOK_DIR / "src").exists() else NOTEBOOK_DIR.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

print(f"Repo root: {REPO_ROOT}")

In [ ]:
import torch
import matplotlib.pyplot as plt

from src.data import build_dataloaders
from src.models import build_model, freeze_backbone
from src.train import (
    _load_env_file,
    build_loss,
    build_optimizer,
    build_scheduler,
    train_from_config,
    train_one_epoch,
    validate_one_epoch,
)
from src.utils import get_device, load_config
from src.visualize import overlay_mask, prediction_mask

device = get_device()
print(f"Using device: {device}")

## Load Config

Uses model specific `yaml` to override defaults for hyperparameters, model weights, architecture setups, etc.

In [ ]:
config_path = REPO_ROOT / "configs" / "resnet34_unet.yaml"
config = load_config(config_path)
config["training"]["loss"].setdefault("ignore_index", config["data"].get("ignore_index", 255))

config

## W&B Auth Check

Local runs read `WANDB_API_KEY` from repo-root `.env`. Colab runs should load it from Colab Secrets in the setup cell above. This cell only reports whether a key is available; it never prints the key.

In [ ]:
_load_env_file()
wandb_enabled = config.get("wandb", {}).get("enabled", False)
wandb_key_available = bool(os.environ.get("WANDB_API_KEY"))

print(f"W&B enabled in config: {wandb_enabled}")
print(f"WANDB_API_KEY available: {wandb_key_available}")
if wandb_enabled and not wandb_key_available:
    print("WARN: NOT LOGGED IN TO W&B.\nAdd WANDB_API_KEY to local .env or Colab Secrets before full training.")

## Data Loading

In [ ]:
# PLACEHOLDER: dataloader integration point. assumes loader lives in src.data and meets contract below.
# Required contract:
#   train_loader, val_loader, test_loader = build_dataloaders(config["data"])
#   images: [B, 5, H, W] float tensor
#   masks: [B, H, W] long tensor with labels 0..4 and optional ignore_index=255

train_loader, val_loader, test_loader = build_dataloaders(config["data"])
len(train_loader), len(val_loader), len(test_loader)

In [ ]:
def unpack_batch(batch):
    if isinstance(batch, dict):
        return batch["image"], batch["mask"]
    return batch

batch = next(iter(train_loader))
images, masks = unpack_batch(batch)

print("images", images.shape, images.dtype, images.min().item(), images.max().item())
print("masks ", masks.shape, masks.dtype, torch.unique(masks)[:20])

## Model, Loss, Optimizer

In [ ]:
model = build_model(config["model"]).to(device)
loss_fn = build_loss(config["training"]["loss"], device)

warmup_frozen_epochs = config["training"].get("warmup_frozen_epochs", 0)
if warmup_frozen_epochs > 0:
    freeze_backbone(model)

optimizer = build_optimizer(model, config["training"]["optimizer"])
scheduler = build_scheduler(
    optimizer,
    config["training"]["scheduler"],
    config["training"]["max_epochs"],
)

print(model.__class__.__name__)
print([group["name"] for group in optimizer.param_groups])

In [ ]:
# Forward-pass smoke test.
model.eval()
with torch.no_grad():
    sample_logits = model(images[:2].to(device))

print("logits", sample_logits.shape)
assert sample_logits.shape[1] == config["model"]["num_classes"]
assert sample_logits.shape[-2:] == images.shape[-2:]

## Single-Batch Verification

Optional. Uncomment when you want to verify one train step and one validation pass. This mutates model weights, so restart the kernel before real training.

In [ ]:
# train_metrics = train_one_epoch(model, [batch], loss_fn, optimizer, device, epoch=1)
# val_metrics = validate_one_epoch(model, [batch], loss_fn, device)
# train_metrics, val_metrics

## Full Training

Optional. Uncomment when the data paths, runtime, and W&B credentials are ready. Checkpoints will be saved under `outputs/resnet34_unet/`.

In [ ]:
# result = train_from_config(config_path)
# result

## Visualize Predictions

In [ ]:
# PLACEHOLDER: run after training or after selecting a checkpoint.
# Missing until a trained checkpoint exists at outputs/resnet34_unet/best.pt.
#
# checkpoint_path = REPO_ROOT / "outputs" / "resnet34_unet" / "best.pt"
# checkpoint = torch.load(checkpoint_path, map_location=device)
# model = build_model(config["model"]).to(device)
# model.load_state_dict(checkpoint["model_state_dict"])
# model.eval()
#
# with torch.no_grad():
#     logits = model(images.to(device))
# pred = prediction_mask(logits[0])
#
# fig, axes = plt.subplots(1, 2, figsize=(10, 5))
# axes[0].imshow(overlay_mask(images[0], masks[0], alpha=0.45))
# axes[0].set_title("Ground Truth")
# axes[0].axis("off")
# axes[1].imshow(overlay_mask(images[0], pred, alpha=0.45))
# axes[1].set_title("Prediction")
# axes[1].axis("off")